In [19]:
import pandas as pd
import numpy as np


: 

: 

In [20]:
%cd /content/Walmart_sales_forecasting
save_file = "data/processed/sales_data_preprocessed.csv"

df_sales = pd.read_csv(save_file, parse_dates=['Date'])
df_sales

: 

: 

: 

In [21]:
df_feature = df_sales.copy()

: 

: 

: 

In [22]:
test_split = pd.Timestamp("2012-08-05")
print(f'Test split day: {test_split}')
df_feature['is_test'] = df_feature['Date'] >= test_split
print(f'Num of test: {df_feature['is_test'].sum()}')
print(f'Num of train: {(~df_feature['is_test']).sum()}')



: 

: 

: 

In [23]:
temp_bins = [-np.inf, 18, 25, 32, np.inf]
temp_label = ['Cold', 'Cool', 'Warm', 'Hot']
df_feature['temp_category'] = pd.cut(df_feature['Temperature'], bins=temp_bins, labels=temp_label)
df_feature

: 

: 

: 

In [24]:
df_feature['total_markdown'] = df_feature[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].sum(axis=1)
df_feature['avg_markdown'] = df_feature[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].mean(axis=1)
df_feature['max_markdown'] = df_feature[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].max(axis=1)

: 

: 

: 

In [25]:
#change to categorical val
df_feature = pd.get_dummies(df_feature, columns=['IsHoliday', 'temp_category'], drop_first=True)
df_feature

: 

: 

: 

In [26]:
# One-hot encoding Type
df_feature = pd.get_dummies(df_feature, columns=['Type'], prefix='Type', drop_first=True)

: 

: 

: 

In [27]:
df_feature = df_feature.sort_values(['Date', 'Store', 'Dept'])
df_feature['store_dept'] = ' store_' + df_feature['Store'].astype(str) + '_' + 'dept_' + df_feature['Dept'].astype(str)
df_feature

: 

: 

: 

In [28]:
lags = [2, 4, 6]
for i in lags:
    df_feature[f'lags_{i}'] = df_feature.groupby('store_dept')['Weekly_Sales'].transform(lambda x : x.shift(i))
df_feature


: 

: 

: 

In [29]:
for window in lags:
    df_feature[f'mean_sales_last_{window}_week'] = df_feature.groupby('Store')['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
    df_feature[f'max_sales_last_{window}_week'] = df_feature.groupby('Store')['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).max())
    df_feature[f'min_sales_last_{window}_week'] = df_feature.groupby('Store')['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).min())
    df_feature[f'std_sales_last_{window}_week'] = df_feature.groupby('Store')['Weekly_Sales'].transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).std())

#exponential moving weight
for alpha in [0.5, 0.75]:
    df_feature[f'emw_sales_{alpha}'] = df_feature.groupby('Store')['Weekly_Sales'].transform(lambda x: x.shift(1).ewm(alpha=alpha, adjust=False).mean())

df_feature

: 

: 

: 

In [30]:
# store level feature
df_feature['sum_store_1_week'] = df_feature.groupby(['Store', 'Date'])['Weekly_Sales'].transform('sum')
df_feature['mean_store_1_week'] = df_feature.groupby(['Store', 'Date'])['Weekly_Sales'].transform('mean')

# dept level feature
df_feature['sum_dept_1_week'] = df_feature.groupby(['Dept', 'Date'])['Weekly_Sales'].transform('sum')
df_feature['mean_dept_1_week'] = df_feature.groupby(['Dept', 'Date'])['Weekly_Sales'].transform('mean')
df_feature

: 

: 

: 

In [31]:
print(df_feature.isna().sum())

: 

: 

: 

In [32]:
nan_count = df_feature.isna().sum()
nan_columns = nan_count[nan_count > 0].index
nan_columns

: 

: 

: 

In [33]:
nan_sample = df_feature.isna().sum(axis=1)
nan_sample = nan_sample[nan_sample > 0].count()
print(f'number of sample containing nan val: {nan_sample}')
print(f'number of total sample: {df_feature['Store'].count()}')
print(f'percent of nan sample: {nan_sample / df_feature['Store'].count()}')

: 

: 

: 

therefore we will choose to drop nan rows

In [34]:

df_feature = df_feature.dropna()
print(f'number of sample after drop: {df_feature['Store'].count()}')

: 

: 

: 

In [35]:
# save data feather format
# feather faster and lighter than csv, no need to parse date like csv
%cd /content/Walmart_sales_forecasting
feather_dir = 'data/processed/feature_engineering.feather'
df_feature.to_feather(feather_dir)
 

: 

: 

: 

In [36]:
%cd /content/Walmart_sales_forecasting/data/processed
!ls

: 

: 

: 